In [1]:

from lume_cheetah import LUMECheetahModel, CheetahSimulator
from lume_cheetah.transformer import SLACCheetahTransformer
from cheetah.accelerator import Segment
from cheetah.particles import ParticleBeam
#from lume_cheetah.mappings.attr_name_mappings import get_mappings
from lume_cheetah.model_configs.loading import variables_from_yaml
from lume_cheetah.utils import get_pv_mad_mapping
import torch
import os

beam_fp = os.path.join("lume_cheetah", "beams", "impact_inj_output_YAG03.h5")
incoming_beam = ParticleBeam.from_openpmd_file(
    path= beam_fp,
    energy=torch.tensor(64e6),
    dtype=torch.float32,
)
#print(incoming_beam)
#print(incoming_beam.energy)
#print(incoming_beam.particle_charges)
incoming_beam.particle_charges = torch.tensor(1.0)
#print(incoming_beam.particle_charges)
lattice_fp = os.path.join("lume_cheetah", "lattices", "nc_hxr.json")
segment = Segment.from_lattice_json( lattice_fp )


# define the simulator
simulator = CheetahSimulator(
    segment=segment,
    initial_beam_distribution=incoming_beam,
)

# define the variables supported by the model
model_config_fp = os.path.join("lume_cheetah", "model_configs", "model_config_nc_injector_DL1.yaml")
control_variables, output_variables = variables_from_yaml(model_config_fp)
cv = { **control_variables, **output_variables}
# define the transformer
mapping_fp = os.path.join("lume_cheetah","mappings", "lcls_elements.csv")
#control_name_to_cheetah = get_mappings(mapping_fp,cv, segment)
#print(control_name_to_cheetah)

#TODO: get original mapping from control system name to madname
#TODO: get list of control system names from variable names
#TODO: get a list of cheetah element names from the segment
#TODO: find the intersection of control system names and cheetah element names
#TODO filter variables to only those that are in the intersection

mapping = get_pv_mad_mapping(mapping_fp)
control_system_names_inp = list(set([name.rsplit(":",1)[0] for name in control_variables.keys()]))
control_system_names_obs = list(set([name.rsplit(":",1)[0] for name in output_variables.keys()]))
cvns = list(set([name.rsplit(":",1)[0] for name in cv.keys()]))
print(mapping)
print(control_system_names_inp)
print(control_system_names_obs)

mapped = {name: mapping[name].lower() for name in cvns if name in mapping}
print(mapped)
element_names = [elem.name for elem in segment.elements]

cvss = list(cv.keys())
for control_name, madname in mapped.items():
    if madname not in element_names:
        for variable_name in cvss:
            if variable_name.startswith(control_name):
                cv.pop(variable_name)

print(cv)


transformer = SLACCheetahTransformer(mapped)


model = LUMECheetahModel(
    simulator=simulator,
    transformer=transformer,
    control_variables=control_variables,
    observable_variables=output_variables,
)




/sdf/home/c/cgarnier/.conda/envs/linac-simulation/lib/python3.11/site-packages/torch/nn/modules/module.py:2066: PhysicsWarning: Invalid tracking method 'cheetah' for element dl00 of type Drift, supported methods are ['linear', 'second_order', 'drift_kick_drift']. Keeping the previous tracking method linear.
  super().__setattr__(name, value)
/sdf/home/c/cgarnier/.conda/envs/linac-simulation/lib/python3.11/site-packages/torch/nn/modules/module.py:2066: PhysicsWarning: Invalid tracking method 'cheetah' for element loadlock of type Drift, supported methods are ['linear', 'second_order', 'drift_kick_drift']. Keeping the previous tracking method linear.
  super().__setattr__(name, value)
/sdf/home/c/cgarnier/.conda/envs/linac-simulation/lib/python3.11/site-packages/torch/nn/modules/module.py:2066: PhysicsWarning: Invalid tracking method 'cheetah' for element dl01a of type Drift, supported methods are ['linear', 'second_order', 'drift_kick_drift']. Keeping the previous tracking method linear

{nan: 'YCSU35', 'SOLN:GUNB:100': 'SOL1BKB', 'CATH:GUNB:100': 'CATHODEB', 'LBLM:GUNB:212:A': 'LBLM00A', 'LBLM:GUNB:212:B': 'LBLM00B', 'QUAD:GUNB:212:1': 'CQ01B', 'SOLN:GUNB:212': 'SOL1B', 'QUAD:GUNB:212:2': 'SQ01B', 'XCOR:GUNB:293': 'XC01B', 'YCOR:GUNB:293': 'YC01B', 'BPMS:GUNB:314': 'BPM1B', 'TORO:GUNB:360': 'IM01B', 'XCOR:GUNB:388': 'XC02B', 'YCOR:GUNB:388': 'YC02B', 'ACCL:GUNB:455': 'BUN1B', 'XCOR:GUNB:513': 'XC03B', 'YCOR:GUNB:513': 'YC03B', 'XCOR:GUNB:713': 'XC04B', 'YCOR:GUNB:713': 'YC04B', 'MOVR:GUNB:753': 'YAG01B', 'QUAD:GUNB:823:1': 'CQ02B', 'SOLN:GUNB:823': 'SOL2B', 'QUAD:GUNB:823:2': 'SQ02B', 'BPMS:GUNB:925': 'BPM2B', 'XCOR:GUNB:927': 'XC05B', 'YCOR:GUNB:927': 'YC05B', 'ACCL:L0B:0110': 'CAVL011', 'ACCL:L0B:0120': 'CAVL012', 'ACCL:L0B:0130': 'CAVL013', 'ACCL:L0B:0140': 'CAVL014', 'ACCL:L0B:0150': 'CAVL015', 'ACCL:L0B:0160': 'CAVL016', 'ACCL:L0B:0170': 'CAVL017', 'ACCL:L0B:0180': 'CAVL018', 'BPMS:L0B:0183': 'CMB01', 'QUAD:L0B:0185': 'QCM01', 'XCOR:L0B:0185': 'XCM01', 'YCOR:L0B:

## Test model setting and getting

In [2]:
mapped

{'XCOR:IN20:521': 'xc07',
 'QUAD:IN20:525': 'qe04',
 'BPMS:IN20:425': 'bpm6',
 'BPMS:IN20:525': 'bpm9',
 'BPMS:IN20:511': 'bpm8',
 'YCOR:IN20:522': 'yc07',
 'QUAD:IN20:441': 'qe02',
 'QUAD:IN20:511': 'qe03',
 'QUAD:IN20:425': 'qe01'}

In [3]:
transformer.get_cheetah_property(simulator, "QUAD:IN20:511:BCTRL")

tensor(8.0429)

In [4]:
control_system_names_inp

['XCOR:IN20:521',
 'QUAD:IN20:525',
 'YCOR:IN20:522',
 'QUAD:IN20:441',
 'QUAD:IN20:511',
 'QUAD:IN20:425']

In [5]:
print(simulator.segment.qe03.k1)
transformer.set_cheetah_property(simulator, "QUAD:IN20:511:BCTRL",value = 1.0)

tensor(11.5680)


In [6]:
print(simulator.segment.qe03.k1)

tensor(1.4383)


In [7]:
transformer.get_cheetah_property(simulator, "QUAD:IN20:511:BCTRL")

tensor(1.)

In [8]:
model.supported_variables
#TODO: need to do something about tmit. (this is handled before passing in variables so should be fine for now,
# but eventually we should allow fo the transformer to handle supported variables and do the filtering there. This way we can have a more flexible transformer that can be used across different simulators and variable naming conventions without having to change the model code. We can also handle any necessary conversions or calculations in the transformer


{'QUAD:IN20:425:BACT': ScalarVariable(name='QUAD:IN20:425:BACT', read_only=False, default_validation_config='none', default_value=-0.19072549045085907, value_range=(-0.22887058854103087, -0.15258039236068727), unit='kG'),
 'QUAD:IN20:425:BCTRL': ScalarVariable(name='QUAD:IN20:425:BCTRL', read_only=False, default_validation_config='none', default_value=-0.19072549045085907, value_range=(-0.22887058854103087, -0.15258039236068727), unit='kG'),
 'QUAD:IN20:441:BACT': ScalarVariable(name='QUAD:IN20:441:BACT', read_only=False, default_validation_config='none', default_value=0.5443543195724487, value_range=(0.435483455657959, 0.6532251834869385), unit='kG'),
 'QUAD:IN20:441:BCTRL': ScalarVariable(name='QUAD:IN20:441:BCTRL', read_only=False, default_validation_config='none', default_value=0.5443543195724487, value_range=(0.435483455657959, 0.6532251834869385), unit='kG'),
 'QUAD:IN20:511:BACT': ScalarVariable(name='QUAD:IN20:511:BACT', read_only=False, default_validation_config='none', defaul

In [ ]:
l = list(model.supported_variables.keys())
values_dict = {name: 1.0 for name in l if name in model.control_variables and 'BACT' not in name}
print(model.get(l))
#set is very slow due to beam energy calculation in transformer, maybe some list format should be passable for args.
print(model.set(values_dict))

{'QUAD:IN20:425:BACT': tensor(-0.1907), 'QUAD:IN20:425:BCTRL': tensor(-0.1907), 'QUAD:IN20:441:BACT': tensor(0.5444), 'QUAD:IN20:441:BCTRL': tensor(0.5444), 'QUAD:IN20:511:BACT': tensor(8.0429), 'QUAD:IN20:511:BCTRL': tensor(8.0429), 'QUAD:IN20:525:BACT': tensor(-4.7208), 'QUAD:IN20:525:BCTRL': tensor(-4.7208), 'XCOR:IN20:521:BACT': tensor(0.), 'XCOR:IN20:521:BCTRL': tensor(0.), 'YCOR:IN20:522:BACT': tensor(0.), 'YCOR:IN20:522:BCTRL': tensor(0.), 'BPMS:IN20:425:TMIT': 1.0, 'BPMS:IN20:425:X': tensor(2.1319e-10), 'BPMS:IN20:425:Y': tensor(-1.0173e-10), 'BPMS:IN20:511:TMIT': 1.0, 'BPMS:IN20:511:X': tensor(6.5023e-09), 'BPMS:IN20:511:Y': tensor(5.0068e-10), 'BPMS:IN20:525:TMIT': 1.0, 'BPMS:IN20:525:X': tensor(4.1151e-09), 'BPMS:IN20:525:Y': tensor(7.7009e-10)}
None


In [12]:
model.get(l)

{'QUAD:IN20:425:BACT': tensor(1.),
 'QUAD:IN20:425:BCTRL': tensor(1.),
 'QUAD:IN20:441:BACT': tensor(1.),
 'QUAD:IN20:441:BCTRL': tensor(1.),
 'QUAD:IN20:511:BACT': tensor(1.),
 'QUAD:IN20:511:BCTRL': tensor(1.),
 'QUAD:IN20:525:BACT': tensor(1.),
 'QUAD:IN20:525:BCTRL': tensor(1.),
 'XCOR:IN20:521:BACT': tensor(1.),
 'XCOR:IN20:521:BCTRL': tensor(1.),
 'YCOR:IN20:522:BACT': tensor(1.),
 'YCOR:IN20:522:BCTRL': tensor(1.),
 'BPMS:IN20:425:TMIT': 1.0,
 'BPMS:IN20:425:X': tensor(2.1274e-10),
 'BPMS:IN20:425:Y': tensor(-1.0371e-10),
 'BPMS:IN20:511:TMIT': 1.0,
 'BPMS:IN20:511:X': tensor(6.6990e-09),
 'BPMS:IN20:511:Y': tensor(4.1982e-10),
 'BPMS:IN20:525:TMIT': 1.0,
 'BPMS:IN20:525:X': tensor(0.0378),
 'BPMS:IN20:525:Y': tensor(0.0382)}

In [ ]:
#does this also need to update? it seems like it should.
model.supported_variables

{'QUAD:IN20:425:BACT': ScalarVariable(name='QUAD:IN20:425:BACT', read_only=False, default_validation_config='none', default_value=-0.19072549045085907, value_range=(-0.22887058854103087, -0.15258039236068727), unit='kG'),
 'QUAD:IN20:425:BCTRL': ScalarVariable(name='QUAD:IN20:425:BCTRL', read_only=False, default_validation_config='none', default_value=-0.19072549045085907, value_range=(-0.22887058854103087, -0.15258039236068727), unit='kG'),
 'QUAD:IN20:441:BACT': ScalarVariable(name='QUAD:IN20:441:BACT', read_only=False, default_validation_config='none', default_value=0.5443543195724487, value_range=(0.435483455657959, 0.6532251834869385), unit='kG'),
 'QUAD:IN20:441:BCTRL': ScalarVariable(name='QUAD:IN20:441:BCTRL', read_only=False, default_validation_config='none', default_value=0.5443543195724487, value_range=(0.435483455657959, 0.6532251834869385), unit='kG'),
 'QUAD:IN20:511:BACT': ScalarVariable(name='QUAD:IN20:511:BACT', read_only=False, default_validation_config='none', defaul

In [14]:
model.reset()

In [ ]:
model.get(l)
# x and y readings became nan.... 

{'QUAD:IN20:425:BACT': tensor(-0.1907),
 'QUAD:IN20:425:BCTRL': tensor(-0.1907),
 'QUAD:IN20:441:BACT': tensor(0.5444),
 'QUAD:IN20:441:BCTRL': tensor(0.5444),
 'QUAD:IN20:511:BACT': tensor(8.0429),
 'QUAD:IN20:511:BCTRL': tensor(8.0429),
 'QUAD:IN20:525:BACT': tensor(-4.7208),
 'QUAD:IN20:525:BCTRL': tensor(-4.7208),
 'XCOR:IN20:521:BACT': tensor(0.),
 'XCOR:IN20:521:BCTRL': tensor(0.),
 'YCOR:IN20:522:BACT': tensor(0.),
 'YCOR:IN20:522:BCTRL': tensor(0.),
 'BPMS:IN20:425:TMIT': 1.0,
 'BPMS:IN20:425:X': tensor(nan),
 'BPMS:IN20:425:Y': tensor(nan),
 'BPMS:IN20:511:TMIT': 1.0,
 'BPMS:IN20:511:X': tensor(nan),
 'BPMS:IN20:511:Y': tensor(nan),
 'BPMS:IN20:525:TMIT': 1.0,
 'BPMS:IN20:525:X': tensor(nan),
 'BPMS:IN20:525:Y': tensor(nan)}